> ⏱ ~10 min

# Scenario

**Section 1.** A travel concierge that calls three specialists is harder to debug than a single agent. When something goes wrong, you want to see the exact path a request took: which specialist the concierge called, which tool that specialist invoked, how long each hop took, and what came back. **Tracing** gives you that record.

Microsoft Agent Framework ships built-in **OpenTelemetry** instrumentation. A **span** is one stopwatch around one piece of work — one agent turn, one tool call, one model request. The spans nest, so a concierge turn becomes a tree that contains its specialist sub-turns and their tool calls.

In this notebook you turn on tracing locally, run the multi-agent concierge once, and look at the spans two ways: printed to your console (always on) and, when `APPLICATIONINSIGHTS_CONNECTION_STRING` is set, sent to Application Insights so the Foundry Monitoring tab lights up just like it did for a Foundry-hosted agent.


## 2. What you will do

1. Call `configure_otel_providers` once to enable MAF telemetry.
2. Add the Azure Monitor exporter when an App Insights connection string is available.
3. Re-run the concierge end to end and print a compact summary of every span.
4. Point at the Foundry Monitoring tab when the cloud exporter is wired up.

Tracing has to be configured **before** the first agent runs in the process. The cleanest pattern is to call it at the top of every entrypoint notebook or script.

---


In [ ]:
# Suppress experimental/deprecation warnings to keep the learning output clean.
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Locate this lab folder portably (do not hardcode the repo name; learners may fork+rename).
LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab-1-foundry-maf':
    for _p in (LAB_DIR, *LAB_DIR.parents):
        _candidate = _p / 'cohere' / 'lab-1-foundry-maf'
        if _candidate.exists():
            LAB_DIR = _candidate
            break
COHERE_DIR = LAB_DIR.parent
load_dotenv(COHERE_DIR / '.env')
sys.path.insert(0, str(LAB_DIR))

APP_INSIGHTS = os.getenv('APPLICATIONINSIGHTS_CONNECTION_STRING') or None
print('App Insights export :', 'enabled' if APP_INSIGHTS else 'disabled (console-only)')


## 3. Configure OpenTelemetry once

`configure_otel_providers` is the MAF helper that wires tracer, meter, and log providers in one call. We always turn on the console exporter so spans are visible even on a laptop, and we attach Azure Monitor when the App Insights connection string is present.

`enable_sensitive_data=True` adds prompt and response text to the spans. That makes the traces far more useful during a workshop. Turn it off in production if your prompts contain personal data.


In [ ]:
from agent_framework.observability import configure_otel_providers

exporters = []
if APP_INSIGHTS:
    # AzureMonitorTraceExporter sends spans to the App Insights instance configured for this project,
    # so the Foundry Monitoring tab shows the local concierge alongside hosted agents.
    from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
    exporters.append(AzureMonitorTraceExporter(connection_string=APP_INSIGHTS))

configure_otel_providers(
    enable_console_exporters=True,
    enable_sensitive_data=True,
    exporters=exporters or None,
)
print('OpenTelemetry configured with', len(exporters), 'cloud exporter(s) and console output.')


## 4. Capture spans into a local collector

The console exporter is great for eyeballing one or two requests, but it floods the notebook quickly. For this lab we also attach a tiny in-memory `SpanExporter` so we can print a clean summary table after the run finishes.

Think of it as a black-box recorder bolted on next to the noisy radio. Both hear the same audio; only one of them keeps a tidy transcript.


In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

memory_exporter = InMemorySpanExporter()
provider = trace.get_tracer_provider()
# configure_otel_providers installs an SDK TracerProvider; we attach an extra processor for the in-memory view.
if isinstance(provider, TracerProvider):
    provider.add_span_processor(SimpleSpanProcessor(memory_exporter))
    print('In-memory span collector attached.')
else:
    print('TracerProvider was not the SDK type; in-memory collection will be skipped.')


## 5. Run the concierge with tracing on

Now build the concierge (after telemetry is configured) and send the same Chicago itinerary request from notebook 02. Every concierge turn, specialist hop, and booking tool call should produce a span.


In [ ]:
from agents import make_chat_client, build_concierge

client = make_chat_client()
concierge = build_concierge(client)

response = await concierge.run(
    'Find a flight, hotel, and standard car for Chicago from 2026-07-18 to 2026-07-21. '
    'Departure city is SFO. Keep everything within corporate policy.'
)
print('--- concierge response ---')
print(response.text)


## 6. Read the spans

Each span has a name, a duration, and a small bag of attributes describing the operation. The next cell prints a compact table: span name, duration in milliseconds, and the parent name when there is one. Lines indented under another span are sub-operations of that span.


In [ ]:
import pandas as pd

spans = list(memory_exporter.get_finished_spans())
rows = []
by_id = {span.context.span_id: span for span in spans}
for span in sorted(spans, key=lambda s: s.start_time or 0):
    parent_id = span.parent.span_id if span.parent else None
    parent_name = by_id[parent_id].name if parent_id in by_id else ''
    duration_ms = ((span.end_time or 0) - (span.start_time or 0)) / 1_000_000
    rows.append({
        'span': span.name,
        'duration_ms': round(duration_ms, 1),
        'parent': parent_name,
        'kind': str(span.kind).split('.')[-1],
    })
pd.DataFrame(rows)


## 7. Open the Foundry Monitoring tab

If `APPLICATIONINSIGHTS_CONNECTION_STRING` was set, the same spans were also sent to Application Insights. Open your Foundry project, switch to the **Monitoring** tab, and filter on the last few minutes — the concierge run shows up as a trace tree with the specialist agents and booking tools nested inside.

Without an App Insights connection string this step is skipped; the console output and the table above are still enough to demonstrate the behavior in a workshop.


## 8. What you confirmed

1. MAF's `configure_otel_providers` turns on tracing for the agent runtime, the chat client, and the function tools in one call.
2. The same spans can flow to the console, an in-memory collector, and Azure Monitor at the same time.
3. The concierge run produces a nested trace: concierge turn → specialist tool calls → specialist turns → booking tool calls.

Next, `04-eval-round1-baseline.ipynb` starts the four-round evaluation arc that scores baseline, grounded, custom-judge, and multi-agent versions of the concierge with `azure-ai-evaluation`.
